[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/burzynski-lab/OU_PHYS2222_ComputationalPhysics/blob/main/LectureNotes/08_Integration.ipynb)

# Numerical Integration

**PHYS 2222: Computational Physics** · University of Oklahoma · Spring 2027 · Week 6

> **Before you start:** in Colab, choose **File → Save a copy in Drive** so your changes are kept.

## Learning goals

By the end of this notebook you should be able to:

- approximate a definite integral with Riemann sums (left and midpoint), the trapezoid rule, and Simpson's rule, and write each one yourself in a few lines of NumPy;
- state the error order of each method and confirm it with a convergence study on a log-log plot;
- explain the idea of Gaussian quadrature and use `np.polynomial.legendre.leggauss`;
- use `scipy.integrate.quad` (including its error estimate and infinite limits), and `scipy.integrate.trapezoid` and `simpson` for sampled data;
- handle integrals over infinite ranges with a change of variables;
- apply all of this to physics: distance from measured velocity, the Stefan–Boltzmann constant, normalizing a wavefunction and computing $\langle x\rangle$, $\Delta x$, and probabilities, and the period of a pendulum at large amplitude.

## 0. Setup

We import `integrate` from SciPy. It is a *submodule*: a library inside the SciPy library. After `from scipy import integrate`, its functions are called as `integrate.quad`, `integrate.simpson`, and so on.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.constants as const
from scipy import integrate

## 1. Integrals as area

The definite integral $I = \int_a^b f(x)\,dx$ is the (signed) area under the curve $f(x)$ between $x = a$ and $x = b$. Many integrals that show up in physics have no formula in terms of elementary functions, for example $\int_0^x e^{-t^2}\,dt$, or the period of a pendulum at large amplitude. Others have a formula, but the integrand comes from *data* (a list of measured numbers), not from a formula at all. In both cases we need to compute the area numerically.

All the methods in this notebook follow the same plan: split $[a, b]$ into $N$ slices of width

$$
h = \frac{b - a}{N},
$$

approximate the area of each slice by a simple shape, and add up the slices.

Our test case, from Newman's book, is

$$
I = \int_0^2 \left(x^4 - 2x + 1\right) dx = \left[\frac{x^5}{5} - x^2 + x\right]_0^2 = 4.4 .
$$

Knowing the exact answer lets us measure the error of each method.

In [ ]:
def f(x):
    return x**4 - 2 * x + 1

a, b = 0.0, 2.0
I_exact = 4.4

## 2. Riemann sums

The simplest shape is a rectangle. You met Riemann sums in calculus when the integral was first defined:

- **Left Riemann sum:** the height of each rectangle is $f$ at the *left* edge of the slice: $\displaystyle I \approx h\sum_{k=0}^{N-1} f(a + kh)$.
- **Midpoint rule:** the height is $f$ at the *middle* of the slice: $\displaystyle I \approx h\sum_{k=0}^{N-1} f\!\left(a + (k + \tfrac12)h\right)$.

With NumPy we make an array of all the sample points at once and add up the function values with `np.sum`. No explicit loop is needed.

In [ ]:
def left_riemann(f, a, b, N):
    """Left Riemann sum with N slices."""
    h = (b - a) / N
    x = a + h * np.arange(N)            # left edges: a, a+h, ..., b-h
    return h * np.sum(f(x))

def midpoint_rule(f, a, b, N):
    """Midpoint rule with N slices."""
    h = (b - a) / N
    x = a + h * (np.arange(N) + 0.5)    # midpoints of the slices
    return h * np.sum(f(x))

In [ ]:
N = 10
print(f"left Riemann: {left_riemann(f, a, b, N):.6f}   error = {left_riemann(f, a, b, N) - I_exact:+.2e}")
print(f"midpoint:     {midpoint_rule(f, a, b, N):.6f}   error = {midpoint_rule(f, a, b, N) - I_exact:+.2e}")

With the same number of function evaluations, the midpoint rule is about 10 times more accurate. A picture shows why. We draw the rectangles with `ax.bar`; `align="edge"` puts each bar's left side at the given $x$, and `alpha` makes them semi-transparent.

In [ ]:
N = 6
h = (b - a) / N
x_fine = np.linspace(a, b, 200)
x_left = a + h * np.arange(N)
x_mid = x_left + h / 2

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
axes[0].bar(x_left, f(x_left), width=h, align="edge", alpha=0.4, edgecolor="k")
axes[0].set_title("left Riemann sum")
axes[1].bar(x_left, f(x_mid), width=h, align="edge", alpha=0.4, edgecolor="k", color="C1")
axes[1].plot(x_mid, f(x_mid), "ko", ms=4)
axes[1].set_title("midpoint rule")
for ax in axes:
    ax.plot(x_fine, f(x_fine), "k-")
    ax.set_xlabel("$x$")
axes[0].set_ylabel("$f(x)$")
plt.show()

In the left sum, every rectangle on a rising curve is too short, and the errors all have the same sign and add up. In the midpoint rule, each rectangle is too high on one side of the midpoint and too low on the other, and these errors mostly cancel. (It is the same reason the central difference beat the forward difference last week.)

## 3. The trapezoid rule

Instead of a flat top, connect $f(x_k)$ and $f(x_{k+1})$ by a straight line. Each slice is then a trapezoid with area $\tfrac{h}{2}\left[f(x_k) + f(x_{k+1})\right]$. Adding all slices, every interior point appears in two trapezoids and the two endpoints appear once:

$$
I \approx h\left[\tfrac12 f(x_0) + f(x_1) + f(x_2) + \cdots + f(x_{N-1}) + \tfrac12 f(x_N)\right], \qquad x_k = a + kh .
$$

Note that there are $N$ slices but $N + 1$ points.

**Error (sketch).** Taylor-expanding $f$ within one slice shows that the straight line misses the curve by an area of about $\tfrac{h^3}{12}f''$ per slice. There are $N = (b-a)/h$ slices, so the total error, defined as $E_\text{trap} = I_\text{exact} - I_\text{trap}$, is about

$$
E_\text{trap} \approx -\frac{h^2}{12}\left[f'(b) - f'(a)\right] = O(h^2).
$$

(The midpoint rule is also $O(h^2)$, with an error about half as big and of opposite sign. The left Riemann sum is only $O(h)$.)

To code this, we need the sum of the interior points `y[1:-1]` (everything except the first and last elements).

In [ ]:
def trapezoid_rule(f, a, b, N):
    """Trapezoid rule with N slices (N + 1 points)."""
    h = (b - a) / N
    x = np.linspace(a, b, N + 1)
    y = f(x)
    return h * (0.5 * y[0] + np.sum(y[1:-1]) + 0.5 * y[-1])

for N in [10, 20, 40]:
    I = trapezoid_rule(f, a, b, N)
    print(f"N = {N:3d}  trapezoid = {I:.8f}   error = {I - I_exact:+.2e}")

Doubling $N$ (halving $h$) cuts the error by 4, as expected for $O(h^2)$. We can also check the error formula: here $f'(x) = 4x^3 - 2$, so $f'(2) - f'(0) = 32$, and with $N = 10$ ($h = 0.2$) the predicted $E_\text{trap} = I_\text{exact} - I_\text{trap}$ is $-\tfrac{0.04}{12}\cdot 32 \approx -0.107$.

In [ ]:
h = 0.2
predicted = (h**2 / 12) * ((4 * 2**3 - 2) - (4 * 0**3 - 2))
print("predicted I_exact - I_trap =", -predicted)
print("actual    I_exact - I_trap =", I_exact - trapezoid_rule(f, a, b, 10))

Size and sign both match. A negative $E_\text{trap}$ means the trapezoid answer is too *big*. Our curve is concave up ($f'' > 0$), so the straight-line tops of the trapezoids lie above the curve. Sign conventions for "error" are easy to mix up, so always check them against a known case like this.

## 4. Simpson's rule

The next step up: instead of a straight line through two points, fit a **parabola** through three points $x_{k-1}, x_k, x_{k+1}$. Integrating that parabola over the two slices it spans gives $\tfrac{h}{3}\left[f(x_{k-1}) + 4f(x_k) + f(x_{k+1})\right]$. Adding up pairs of slices gives **Simpson's rule**:

$$
I \approx \frac{h}{3}\Big[f(x_0) + 4f(x_1) + 2f(x_2) + 4f(x_3) + \cdots + 2f(x_{N-2}) + 4f(x_{N-1}) + f(x_N)\Big].
$$

The weights go $1, 4, 2, 4, 2, \ldots, 4, 1$: odd-numbered points get 4, interior even-numbered points get 2. Because the slices are used in pairs, **$N$ must be even**.

The error turns out to be

$$
E_\text{Simp} = I_\text{exact} - I_\text{Simp} \approx -\frac{h^4}{180}\left[f^{(3)}(b) - f^{(3)}(a)\right] = O(h^4),
$$

two powers of $h$ better than the trapezoid rule. (Surprisingly, Simpson's rule is exact for cubics, not just parabolas, thanks to a symmetry cancellation.)

To pick out the odd and even points, we use slicing with a **step**: `y[start:stop:step]`. For example, `y[1:-1:2]` means "start at index 1, stop before the last element, take every 2nd element", which is indices 1, 3, 5, ... .

In [ ]:
y = np.arange(11)
print("all:            ", y)
print("odd  y[1:-1:2]: ", y[1:-1:2])
print("even y[2:-1:2]: ", y[2:-1:2])

In [ ]:
def simpson_rule(f, a, b, N):
    """Simpson's rule with N slices (N must be even)."""
    if N % 2 != 0:
        raise ValueError("N must be even for Simpson's rule")
    h = (b - a) / N
    x = np.linspace(a, b, N + 1)
    y = f(x)
    return h / 3 * (y[0] + y[-1] + 4 * np.sum(y[1:-1:2]) + 2 * np.sum(y[2:-1:2]))

for N in [10, 20, 40]:
    I = simpson_rule(f, a, b, N)
    print(f"N = {N:3d}  Simpson = {I:.10f}   error = {I - I_exact:+.2e}")

Doubling $N$ now cuts the error by $2^4 = 16$. With just $N = 10$, Simpson's rule is 1000 times more accurate than the trapezoid rule, for the same number of function evaluations.

The `raise ValueError(...)` line stops the function with an error message if someone passes an odd `N`. It is good practice to check inputs like this. Here is what the user would see (we catch the error with `try`/`except` so this notebook keeps running):

In [ ]:
try:
    simpson_rule(f, a, b, 7)
except ValueError as err:
    print("ValueError:", err)

### Your turn 1

Compute $\int_0^{\pi} \sin x\,dx$ (exact answer 2) with `left_riemann`, `midpoint_rule`, `trapezoid_rule`, and `simpson_rule`, each for $N = 8$ and $N = 16$. For each method, compute the ratio error($N=8$) / error($N=16$). What ratio does each error order ($O(h)$, $O(h^2)$, $O(h^4)$) predict?

In [ ]:
# Your code here

## 5. Convergence study

To compare methods properly, we compute the error for many values of $N$ and plot error versus $N$ on log-log axes. Since $h \propto 1/N$, an error $\propto h^n$ is a straight line with slope $-n$.

In [ ]:
Ns = 2 ** np.arange(1, 21)          # 2, 4, 8, ..., about a million
methods = {"left Riemann": left_riemann, "midpoint": midpoint_rule,
           "trapezoid": trapezoid_rule, "Simpson": simpson_rule}

errors = {}
for name, rule in methods.items():
    errors[name] = np.array([abs(rule(f, a, b, N) - I_exact) for N in Ns])

Two new pieces of syntax here:

- `methods` is a **dictionary** of functions. Looping over `methods.items()` gives each name and function in turn.
- `[... for N in Ns]` is a **list comprehension**: a compact way of writing a `for` loop that builds a list. `[g(N) for N in Ns]` means "the list of `g(N)` for every `N` in `Ns`".

In [ ]:
Nf = Ns.astype(float)    # float copy of Ns, for the reference lines

fig, ax = plt.subplots(figsize=(7, 5))
for name in methods:
    ax.loglog(Ns, errors[name], "o-", ms=4, label=name)
ax.loglog(Nf, 5 / Nf, "k--", lw=1, label=r"$\propto N^{-1}$")
ax.loglog(Nf, 5 / Nf**2, "k:", lw=1, label=r"$\propto N^{-2}$")
ax.loglog(Nf, 5 / Nf**4, "k-.", lw=1, label=r"$\propto N^{-4}$")
ax.set_xlabel("number of slices $N$")
ax.set_ylabel("absolute error")
ax.set_ylim(1e-17, 1e2)
ax.legend(fontsize=8)
plt.show()

Things to notice:

1. Each method follows its predicted slope: $-1$ for left Riemann, $-2$ for midpoint and trapezoid, $-4$ for Simpson.
2. Simpson's error stops decreasing at about $10^{-15}$, near $N \approx 10^4$. That is **round-off error** again: the answer, 4.4, can only be stored to a relative precision of $\varepsilon \approx 2\times10^{-16}$, so no method can do better than about $10^{-15}$. Some Simpson points are missing from the plot because their error came out as exactly 0 (a lucky rounding), and $\log 0$ cannot be plotted. For less friendly integrands, round-off errors accumulate in the sum and the floor is higher.
3. For a smooth integrand, Simpson reaches $10^{-10}$ with a few hundred slices, while the trapezoid rule needs a few hundred *thousand*.

**Pitfall:** we drew the reference lines with `Nf`, a float copy of `Ns`. NumPy integers have a fixed size (64 bits), so `Ns**4` would *overflow* for the largest `N` ($2^{80} > 2^{63}$) and silently give a wrong answer. Python floats go up to about $10^{308}$.

### 5.1 Richardson extrapolation, again

Last week we combined two central differences to cancel the leading error. The same trick works here. The trapezoid error is $c\,h^2 + O(h^4)$, so if $T_N$ is the trapezoid result with $N$ slices,

$$
I \approx T_{2N} + \frac{T_{2N} - T_N}{3}.
$$

The difference $T_{2N} - T_N$ also gives a practical **error estimate** for $T_{2N}$ when you don't know the exact answer: $I - T_{2N} \approx (T_{2N} - T_N)/3$.

In [ ]:
T10 = trapezoid_rule(f, a, b, 10)
T20 = trapezoid_rule(f, a, b, 20)
print("estimated I - T20:", (T20 - T10) / 3)
print("actual    I - T20:", I_exact - T20)
print("Richardson result:",  T20 + (T20 - T10) / 3)
print("Simpson, N = 20:  ",  simpson_rule(f, a, b, 20))

The Richardson-improved trapezoid rule is *identical* to Simpson's rule. Repeating this idea to cancel the $h^4$, $h^6$, ... terms is called **Romberg integration**.

## 6. Gaussian quadrature (the idea)

All the methods so far use **equally spaced** points. If you are free to choose *where* to evaluate $f$, you can do much better. **Gaussian quadrature** writes

$$
\int_{-1}^{1} f(x)\,dx \approx \sum_{k=1}^{N} w_k\, f(x_k),
$$

with the $N$ points $x_k$ and weights $w_k$ chosen so the formula is *exact* for every polynomial of degree up to $2N - 1$. (The points turn out to be the zeros of the Legendre polynomial $P_N(x)$, which you may meet again in quantum mechanics.) NumPy computes them for you:

In [ ]:
xk, wk = np.polynomial.legendre.leggauss(3)
print("points: ", xk)
print("weights:", wk)

These are for the interval $[-1, 1]$. For a general interval $[a, b]$ we stretch and shift them: $x \to \tfrac{b-a}{2}x + \tfrac{b+a}{2}$, and multiply the weights by $\tfrac{b-a}{2}$.

In [ ]:
def gauss_legendre(f, a, b, N):
    """Gaussian quadrature with N points on [a, b]."""
    xk, wk = np.polynomial.legendre.leggauss(N)
    x = 0.5 * (b - a) * xk + 0.5 * (b + a)
    w = 0.5 * (b - a) * wk
    return np.sum(w * f(x))

for N in [2, 3, 4]:
    I = gauss_legendre(f, a, b, N)
    print(f"N = {N} points: {I:.15f}   error = {I - I_exact:+.1e}")

With only **3 points**, Gaussian quadrature integrates our degree-4 polynomial exactly (to round-off), because 3 points handle degree up to $2\cdot 3 - 1 = 5$. For smooth non-polynomial functions it also converges extremely fast. The catch: you must be able to evaluate $f$ at the special points, so it is for integrating *functions*, not equally spaced *data*. The workhorse SciPy routine below uses a refined version of this idea.

### Your turn 2

Use `gauss_legendre` to compute $\int_0^{\pi}\sin x\,dx$ for $N = 2, 3, \ldots, 8$ points and print the error each time. How many points do you need to reach an error below $10^{-10}$? How many slices would Simpson's rule need for the same accuracy (use the plot or experiment)?

In [ ]:
# Your code here

## 7. Integration with SciPy

In practice, you will usually call a library routine. `scipy.integrate` offers two families:

| You have... | Use | Notes |
|---|---|---|
| a **function** you can evaluate anywhere | `integrate.quad(f, a, b)` | adaptive Gaussian quadrature; returns value **and** error estimate |
| **sampled data** `y` at points `x` | `integrate.trapezoid(y, x=x)`, `integrate.simpson(y, x=x)` | the rules from Sections 3–4 |

### 7.1 `quad`

`quad` returns two numbers: the integral and an estimate of its absolute error. We **unpack** them into two variables in one line.

In [ ]:
value, err_est = integrate.quad(f, a, b)
print(f"quad result    = {value:.15f}")
print(f"error estimate = {err_est:.1e}")
print(f"actual error   = {value - I_exact:.1e}")

`quad` is **adaptive**: it estimates the error in each part of the interval and puts more points where the integrand is hard. Always look at the error estimate; if it is large, `quad` will usually also print a warning.

Often you want to integrate a function that has extra parameters, like $e^{-\alpha x^2}$. A handy way to make a quick one-line function is a **`lambda`**: `lambda x: np.exp(-alpha * x**2)` is a function of `x`, equivalent to a `def` with a single `return` line.

In [ ]:
alpha = 2.0
gaussian = lambda x: np.exp(-alpha * x**2)
print(gaussian(0.0), gaussian(1.0))

value, err_est = integrate.quad(gaussian, -1, 1)
print(value, err_est)

### 7.2 Infinite limits

`quad` accepts `np.inf` (infinity) or `-np.inf` as limits. Let's check the Gaussian integral $\int_{-\infty}^{\infty} e^{-\alpha x^2}dx = \sqrt{\pi/\alpha}$.

In [ ]:
value, err_est = integrate.quad(gaussian, -np.inf, np.inf)
print("quad:  ", value, "+/-", err_est)
print("exact: ", np.sqrt(np.pi / alpha))

How can a computer integrate to infinity? Internally, `quad` uses a **change of variables** that maps the infinite range onto a finite one. You can do this yourself with any rule. A common choice is

$$
x = \frac{z}{1 - z}, \qquad dx = \frac{dz}{(1-z)^2},
$$

which maps $z \in [0, 1)$ onto $x \in [0, \infty)$:

$$
\int_0^\infty f(x)\,dx = \int_0^1 \frac{f\!\left(\frac{z}{1-z}\right)}{(1-z)^2}\,dz .
$$

The new integrand is evaluated at $z = 1$ only as a limit, so a rule that does not use the endpoints, like the midpoint rule, is convenient. Let's check with $\int_0^\infty e^{-\alpha x^2}dx = \tfrac12\sqrt{\pi/\alpha}$.

In [ ]:
def g_z(z):
    x = z / (1 - z)
    return gaussian(x) / (1 - z)**2

for N in [10, 100, 1000]:
    print(N, midpoint_rule(g_z, 0, 1, N))
print("exact:", 0.5 * np.sqrt(np.pi / alpha))

### 7.3 Sampled data: `trapezoid` and `simpson`

When you only have numbers, not a function, use `integrate.trapezoid(y, x=x)` or `integrate.simpson(y, x=x)`. The keyword argument `x=x` gives the sample positions (they need not be evenly spaced).

In [ ]:
x_samples = np.linspace(a, b, 21)        # 21 points = 20 slices
y_samples = f(x_samples)
print("trapezoid:", integrate.trapezoid(y_samples, x=x_samples))
print("simpson:  ", integrate.simpson(y_samples, x=x_samples))
print("exact:    ", I_exact)

These agree with our own `trapezoid_rule` and `simpson_rule` for $N = 20$. (Older code uses `np.trapz` or `integrate.simps`; those names are deprecated, so use `trapezoid` and `simpson`.)

## 8. Real data: distance from velocity

The file `velocities.txt` (Newman, *Computational Physics*, Exercise 5.1) contains the $x$-velocity of a particle measured once per second: column 0 is time (s), column 1 is velocity (m/s). Last week we differentiated it to get acceleration. Now we integrate it to get position:

$$
x(t) = x(0) + \int_0^t v(t')\,dt'.
$$

In [ ]:
import os
DATA_URL = "https://raw.githubusercontent.com/burzynski-lab/OU_PHYS2222_ComputationalPhysics/main/Data/"

def data_path(filename):
    """Return a local path to a course data file if it exists, otherwise its GitHub URL."""
    for folder in ["../Data", "Data", "."]:
        path = os.path.join(folder, filename)
        if os.path.exists(path):
            return path
    return DATA_URL + filename

In [ ]:
data = np.loadtxt(data_path("velocities.txt"))
t_data, v_data = data[:, 0], data[:, 1]     # time (s), velocity (m/s)

disp_trap = integrate.trapezoid(v_data, x=t_data)
disp_simp = integrate.simpson(v_data, x=t_data)
print(f"net displacement (trapezoid): {disp_trap:.3f} m")
print(f"net displacement (Simpson):   {disp_simp:.3f} m")

The two methods agree to about 3 digits, which tells you the integral is not very sensitive to the method here.

To get $x(t)$ at *every* time, not just the final displacement, we need a running total of the trapezoid areas. `integrate.cumulative_trapezoid` does exactly that. `initial=0` sets $x(0) = 0$ and makes the output the same length as the input.

In [ ]:
x_data = integrate.cumulative_trapezoid(v_data, x=t_data, initial=0)   # position (m)

fig, axes = plt.subplots(2, 1, figsize=(7, 6), sharex=True)
axes[0].plot(t_data, v_data, ".-")
axes[0].axhline(0, color="gray", lw=0.8)
axes[0].set_ylabel("velocity (m/s)")
axes[1].plot(t_data, x_data, ".-", color="C1")
axes[1].set_ylabel("position (m)")
axes[1].set_xlabel("time (s)")
plt.show()

Notice that the position has a maximum or minimum exactly where the velocity crosses zero.

The *net displacement* counts backward motion as negative. The total **distance traveled** (what an odometer would read) is $\int |v|\,dt$:

In [ ]:
distance = integrate.trapezoid(np.abs(v_data), x=t_data)
print(f"total distance traveled: {distance:.3f} m")

Compare this with last week. Differentiating data **amplifies** noise; integrating data **averages** it out, because random positive and negative errors tend to cancel in the sum. Integration is the numerically "friendly" operation.

## 9. Physics: the Stefan–Boltzmann constant

**Physics recap.** A **blackbody** is an ideal object that absorbs all light falling on it; when hot, it glows with a spectrum that depends only on its temperature $T$. Planck explained this spectrum by assuming light comes in quanta (photons) of energy $E = h\nu$. The power radiated per unit area, per unit frequency, is

$$
I(\nu, T) = \frac{2\pi h\nu^3}{c^2}\,\frac{1}{e^{h\nu/k_BT} - 1}.
$$

The **total** power per unit area is the integral over all frequencies. Experimentally it is $\sigma T^4$ (the Stefan–Boltzmann law), with $\sigma \approx 5.67\times10^{-8}\ \mathrm{W\,m^{-2}\,K^{-4}}$. Changing variables to the dimensionless $x = h\nu/k_BT$ gives

$$
\int_0^\infty I\,d\nu = \underbrace{\frac{2\pi k_B^4}{h^3c^2}\int_0^\infty \frac{x^3}{e^x - 1}\,dx}_{\sigma}\;T^4 .
$$

So computing $\sigma$ from fundamental constants comes down to one dimensionless integral. (Its exact value is $\pi^4/15$, but let's pretend we don't know that.)

**Numerical care.** Two details: at $x = 0$ the integrand is $0/0$ (its limit is 0), and for large $x$, $e^x$ overflows to `inf` (around $x \approx 710$). Multiplying the top and bottom by $e^{-x}$ gives the equivalent form $x^3 e^{-x}/(1 - e^{-x})$, which is safe at large $x$. `np.expm1(y)` computes $e^y - 1$ accurately even for tiny $y$ (it avoids the catastrophic cancellation from notebook 06), so $1 - e^{-x}$ is `-np.expm1(-x)`.

In [ ]:
def planck_integrand(x):
    """x^3 / (e^x - 1), written in an overflow-safe way."""
    return x**3 * np.exp(-x) / (-np.expm1(-x))

J, J_err = integrate.quad(planck_integrand, 0, np.inf)
print(f"integral = {J:.12f} +/- {J_err:.1e}")
print(f"pi^4/15  = {np.pi**4 / 15:.12f}")

In [ ]:
sigma_num = 2 * np.pi * const.k**4 / (const.h**3 * const.c**2) * J
print(f"sigma (our integral)    = {sigma_num:.6e} W m^-2 K^-4")
print(f"sigma (scipy.constants) = {const.sigma:.6e} W m^-2 K^-4")
print(f"relative difference     = {abs(sigma_num - const.sigma) / const.sigma:.1e}")

We reproduced a fundamental constant to about 10 significant figures from $h$, $c$, $k_B$ and one integral.

As a cross-check, use the change of variables from Section 7.2 with the midpoint rule:

In [ ]:
def planck_z(z):
    x = z / (1 - z)
    return planck_integrand(x) / (1 - z)**2

print("midpoint, N = 1000:", midpoint_rule(planck_z, 0, 1, 1000))

## 10. Physics: wavefunctions and probabilities

**Physics recap.** In quantum mechanics, a particle in one dimension is described by a wavefunction $\psi(x)$. The probability of finding the particle between $x_1$ and $x_2$ is

$$
P(x_1 \le x \le x_2) = \int_{x_1}^{x_2} |\psi(x)|^2\,dx .
$$

Because the particle must be *somewhere*, a physical wavefunction is **normalized**: $\int_{-\infty}^{\infty}|\psi|^2\,dx = 1$. The average position (expectation value) and its spread (uncertainty) are

$$
\langle x\rangle = \int x\,|\psi|^2\,dx, \qquad \langle x^2\rangle = \int x^2\,|\psi|^2\,dx, \qquad \Delta x = \sqrt{\langle x^2\rangle - \langle x\rangle^2}.
$$

Every one of these is an integral.

### 10.1 Particle in a box

For a particle trapped in an infinite square well ("box") of width $L$, the ground state is $\psi(x) = A\sin(\pi x/L)$ for $0 \le x \le L$ and zero outside. Let's find $A$ numerically, for an electron in a box of width $L = 1$ nm. We work in nm.

In [ ]:
L = 1.0                                  # box width (nm)

def psi_unnorm(x):
    return np.sin(np.pi * x / L)

norm2, _ = integrate.quad(lambda x: psi_unnorm(x)**2, 0, L)
A = 1 / np.sqrt(norm2)
print(f"integral of |psi|^2 before normalizing = {norm2:.6f} nm")
print(f"A = {A:.6f} nm^-1/2   (analytic: sqrt(2/L) = {np.sqrt(2 / L):.6f})")

The underscore `_` in `norm2, _ = ...` is a common Python convention for "a value I don't need" (here, the error estimate).

Now the normalized wavefunction and the expectation values:

In [ ]:
def prob_density(x):
    return (A * psi_unnorm(x))**2

x_mean, _ = integrate.quad(lambda x: x * prob_density(x), 0, L)
x2_mean, _ = integrate.quad(lambda x: x**2 * prob_density(x), 0, L)
dx_unc = np.sqrt(x2_mean - x_mean**2)

print(f"<x>    = {x_mean:.6f} nm      (analytic: L/2 = {L / 2})")
print(f"<x^2>  = {x2_mean:.6f} nm^2   (analytic: {L**2 * (1/3 - 1 / (2 * np.pi**2)):.6f})")
print(f"Delta x = {dx_unc:.6f} nm")

$\langle x\rangle = L/2$ makes sense: the probability density is symmetric about the center. The uncertainty $\Delta x \approx 0.18\,L$ tells you how spread out the electron is.

What is the probability of finding the electron in the left quarter of the box, $0 \le x \le L/4$? (A classical particle bouncing back and forth at constant speed would be there 25% of the time.)

In [ ]:
P_quarter, _ = integrate.quad(prob_density, 0, L / 4)
print(f"P(0 < x < L/4) = {P_quarter:.4f}")
print(f"analytic:        {1/4 - 1 / (2 * np.pi):.4f}")

Only about 9%: the quantum ground state piles up probability in the middle of the box.

### 10.2 Harmonic oscillator: the classically forbidden region

**Physics recap.** A classical mass on a spring with energy $E$ moves between two **turning points** where all the energy is potential: $\tfrac12 m\omega^2 x^2 = E$. It can never go beyond them. A quantum particle can: its wavefunction leaks into the **classically forbidden region**. (This is the same physics as tunneling.)

In natural units ($\hbar = m = \omega = 1$, as in last week's notebook), the ground state is $\psi_0(x) = \pi^{-1/4}e^{-x^2/2}$ with energy $E_0 = \tfrac12$. The turning points are where $\tfrac12 x^2 = \tfrac12$, so $x = \pm 1$. The probability of finding the particle outside them is

$$
P_\text{forbidden} = 2\int_1^{\infty}|\psi_0(x)|^2\,dx
$$

(the factor 2 accounts for both sides, since $|\psi_0|^2$ is symmetric).

In [ ]:
def psi0(x):
    """Harmonic oscillator ground state (natural units)."""
    return np.pi**(-0.25) * np.exp(-x**2 / 2)

total, _ = integrate.quad(lambda x: psi0(x)**2, -np.inf, np.inf)
P_out, P_err = integrate.quad(lambda x: psi0(x)**2, 1, np.inf)
P_forbidden = 2 * P_out
print(f"normalization check: {total:.12f}")
print(f"P(forbidden) = {P_forbidden:.6f}")

About **16%** of the time, a quantum oscillator in its ground state is found where a classical one could never be. This integral has no elementary closed form, but it can be written with the *complementary error function*, which SciPy provides in `scipy.special`: $P = \operatorname{erfc}(1)$.

In [ ]:
from scipy import special
print(f"erfc(1) = {special.erfc(1.0):.6f}")

In [ ]:
x_plot = np.linspace(-3.5, 3.5, 400)
forbidden = np.abs(x_plot) > 1

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(x_plot, psi0(x_plot)**2, "k-", label=r"$|\psi_0(x)|^2$")
ax.fill_between(x_plot, psi0(x_plot)**2, where=forbidden, color="C3", alpha=0.4,
                label=f"forbidden region, P = {P_forbidden:.3f}")
ax.axvline(-1, color="gray", ls="--", lw=1)
ax.axvline(1, color="gray", ls="--", lw=1)
ax.set_xlabel(r"$x$ (units of $\sqrt{\hbar/m\omega}$)")
ax.set_ylabel(r"probability density")
ax.legend()
plt.show()

`ax.fill_between(x, y, where=mask)` shades the area under `y` only where the boolean array `mask` is `True`.

### Your turn 3

The first excited state of the harmonic oscillator (natural units) is $\psi_1(x) = N_1\,x\,e^{-x^2/2}$, with energy $E_1 = \tfrac32$.

1. Find the normalization constant $N_1$ with `quad` (compare with the exact value $\sqrt{2}\,\pi^{-1/4}$).
2. Compute $\langle x\rangle$, $\langle x^2\rangle$, and $\Delta x$. (Expected: $\langle x^2\rangle = 3/2$.)
3. Find the classical turning points for $E_1$ and compute the probability of finding the particle in the forbidden region. Is it bigger or smaller than for the ground state?

In [ ]:
# Your code here

## 11. Physics: the period of a pendulum at large amplitude

**Physics recap.** For small swings, a simple pendulum of length $\ell$ is a harmonic oscillator with period $T_0 = 2\pi\sqrt{\ell/g}$, independent of the amplitude. That approximation uses $\sin\theta \approx \theta$. For large amplitudes $\theta_0$, energy conservation is exact: the kinetic energy at angle $\theta$ equals the loss of potential energy from the highest point,

$$
\tfrac12 m\ell^2\dot\theta^2 = mg\ell\,(\cos\theta - \cos\theta_0)
\quad\Rightarrow\quad
\frac{d\theta}{dt} = \sqrt{\frac{2g}{\ell}(\cos\theta - \cos\theta_0)} .
$$

Solving for $dt$ and integrating over a quarter swing (from $\theta = 0$ to $\theta_0$) gives

$$
T = 4\sqrt{\frac{\ell}{g}}\int_0^{\theta_0}\frac{d\theta}{\sqrt{2(\cos\theta - \cos\theta_0)}} .
$$

The integrand **diverges** at $\theta = \theta_0$, where the pendulum stops for an instant. The integral is still finite (the divergence is like $1/\sqrt{\theta_0 - \theta}$), and `quad` can handle it because it never evaluates exactly at the endpoint. Simple rules like Simpson's, which use the endpoint, would fail.

In [ ]:
def period_ratio(theta0):
    """T / T0 for a pendulum of amplitude theta0 (radians)."""
    integrand = lambda th: 1 / np.sqrt(2 * (np.cos(th) - np.cos(theta0)))
    value, _ = integrate.quad(integrand, 0, theta0)
    return 4 * value / (2 * np.pi)         # T0 = 2 pi sqrt(l/g)

for deg in [5, 20, 45, 90, 135, 170]:
    print(f"amplitude {deg:3d} deg:  T/T0 = {period_ratio(np.radians(deg)):.5f}")

At 5° the small-angle formula is accurate to 0.05%; at 90° the true period is 18% longer, and as $\theta_0 \to 180°$ (balancing upside down) the period grows without limit.

A substitution, $\sin\phi = \sin(\theta/2)/\sin(\theta_0/2)$, removes the divergence and turns the integral into the smooth **complete elliptic integral of the first kind**, $T = 4\sqrt{\ell/g}\,K(k)$ with $k = \sin(\theta_0/2)$ and

$$
K(k) = \int_0^{\pi/2}\frac{d\phi}{\sqrt{1 - k^2\sin^2\phi}} .
$$

Removing an endpoint singularity with a substitution like this is a standard trick: the new integrand is smooth, so even simple rules work well.

In [ ]:
def K_elliptic(k, N=20):
    return simpson_rule(lambda phi: 1 / np.sqrt(1 - k**2 * np.sin(phi)**2), 0, np.pi / 2, N)

theta0 = np.radians(90)
print("T/T0 from smooth form, Simpson N=20:", 4 * K_elliptic(np.sin(theta0 / 2)) / (2 * np.pi))
print("T/T0 from singular form, quad:      ", period_ratio(theta0))

### Your turn 4

Plot $T/T_0$ versus amplitude $\theta_0$ from $1°$ to $179°$. On the same axes, plot the first correction from a series expansion, $T/T_0 \approx 1 + \theta_0^2/16$ (with $\theta_0$ in radians). Up to what amplitude is the small-angle result $T = T_0$ good to 1%? Up to what amplitude is the corrected formula good to 1%? Label your axes, with units.

In [ ]:
# Your code here

### Your turn 5 (Newman Exercise 5.3)

The function

$$
E(x) = \int_0^x e^{-t^2}\,dt
$$

cannot be written in terms of elementary functions, but it appears all over physics and statistics (it is the integral of a Gaussian).

1. Write a function `E(x)` that computes it with your `simpson_rule` and $N = 100$ slices.
2. Evaluate it for `x` from 0 to 3 in steps of 0.1 (use a loop or a list comprehension) and plot $E(x)$.
3. Compare with SciPy's error function: $E(x) = \tfrac{\sqrt{\pi}}{2}\,\operatorname{erf}(x)$, available as `special.erf(x)`. Plot the difference. How large is it?

In [ ]:
# Your code here

## Summary

| Method | Idea | Error | Notes |
|---|---|---|---|
| Left Riemann | rectangles, left edge | $O(h)$ | for understanding only |
| Midpoint | rectangles, middle | $O(h^2)$ | never evaluates endpoints |
| Trapezoid | straight lines | $O(h^2)$ | `integrate.trapezoid(y, x=x)` |
| Simpson | parabolas, $N$ even | $O(h^4)$ | `integrate.simpson(y, x=x)` |
| Gaussian quadrature | optimal points & weights | exact to degree $2N-1$ | `np.polynomial.legendre.leggauss` |
| Adaptive | refine where needed | given by error estimate | `integrate.quad(f, a, b)` |

- A **convergence study** (error vs $N$ on log-log axes) verifies the error order; round-off sets a floor at very large $N$.
- **Richardson extrapolation** of the trapezoid rule gives Simpson's rule; $(T_{2N} - T_N)/3$ estimates the error.
- `integrate.quad` returns `(value, error_estimate)`; it accepts `np.inf` limits and integrable endpoint singularities.
- **Infinite ranges:** substitute $x = z/(1-z)$; **endpoint singularities:** find a substitution that smooths the integrand.
- `integrate.cumulative_trapezoid(y, x=x, initial=0)` gives a running integral, e.g. position from velocity.
- Integrating data averages noise down; differentiating data amplifies it.
- New Python: submodules (`from scipy import integrate`), slicing with a step `y[1:-1:2]`, dictionaries of functions, list comprehensions, `lambda`, unpacking `value, err = ...`, `_` for unused values, `raise`.

## Further reading

- M. Newman, *Computational Physics*, Chapter 5 (integrals), free chapters at <https://websites.umich.edu/~mejn/cp/>.
- SciPy documentation: [Integration tutorial (`scipy.integrate`)](https://docs.scipy.org/doc/scipy/tutorial/integrate.html).
- SciPy documentation: [`scipy.special`](https://docs.scipy.org/doc/scipy/reference/special.html) for erf, elliptic integrals, and other special functions.